In [1]:
import json
import re
import fitz  # PyMuPDF
from typing import List, Dict, Optional
from langchain_text_splitters import RecursiveCharacterTextSplitter


# =========================
# 1. PDF EXTRACTION
# =========================
def extract_text_with_fitz(pdf_path: str) -> str:
    doc = fitz.open(pdf_path)
    pages = [page.get_text("text") for page in doc]
    return "\n".join(pages)


# =========================
# 2. CLEANING
# =========================
def clean_legal_pdf_text(text: str) -> str:
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    text = re.sub(r"CÔNG BÁO/Số\s+\d+\s*\+\s*\d+/Ngày\s+\d{1,2}-\d{1,2}-\d{4}", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"\n\s*\d{1,3}\s*\n", "\n", text)
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


# =========================
# 3. PARSING CHAPTERS + ARTICLES
# =========================
CHAPTER_REGEX = re.compile(r"(Chương\s+[IVXLCDM]+)\s*\n+\s*([^\n]+)", flags=re.UNICODE)
ARTICLE_REGEX = re.compile(r"(Điều\s+\d+[a-z]?\.\s*[^\n]+)", flags=re.UNICODE)


def parse_chapters(text: str) -> List[Dict]:
    matches = list(CHAPTER_REGEX.finditer(text))
    chapters = []

    if not matches:
        return [{"chapter_label": None, "chapter_title": None, "chapter_full": None, "content": text}]

    for i, match in enumerate(matches):
        end = matches[i + 1].start() if i + 1 < len(matches) else len(text)
        chapter_label = match.group(1).strip()
        chapter_title = match.group(2).strip()
        chapter_full = f"{chapter_label} {chapter_title}"
        content = text[match.end():end].strip()

        chapters.append({
            "chapter_label": chapter_label,
            "chapter_title": chapter_title,
            "chapter_full": chapter_full,
            "content": content
        })
    return chapters


def split_article_blocks(chapter_content: str) -> List[Dict]:
    matches = list(ARTICLE_REGEX.finditer(chapter_content))
    articles = []

    if not matches:
        return []

    for i, match in enumerate(matches):
        article_title = match.group(1).strip()
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(chapter_content)
        article_content = chapter_content[start:end].strip()

        articles.append({
            "article_title": article_title,
            "article_content": article_content
        })
    return articles


# =========================
# 4. SPLIT LONG ARTICLES (ĐÃ SỬA - ÍT TÁCH HƠN)
# =========================
def split_long_article(article_text: str, max_length: int = 2500) -> List[str]:
    """
    Chỉ tách khi Điều quá dài.
    Ưu tiên giữ toàn bộ Điều làm 1 chunk.
    Nếu quá dài thì tách theo nhóm lớn (không tách từng khoản 1., 2., 3...).
    """
    if len(article_text) <= max_length:
        return [article_text.strip()]

    # Nếu quá dài, dùng splitter với separator lớn hơn
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=max_length,
        chunk_overlap=200,
        separators=["\n\n\n", "\n\n", "\n1\. ", "\n2\. ", "\n3\. ", "\n", ". ", " ", ""]
    )
    return splitter.split_text(article_text)


# =========================
# 5. FULL PROCESSING (ĐÃ SỬA)
# =========================
def process_legal_pdf(
    pdf_path: str,
    source_file: str,
    law_name: Optional[str] = None,
    domain: Optional[str] = None,
    effective_date: Optional[str] = None,
    law_id: Optional[str] = None,
    max_context_len: int = 2500   # TĂNG LÊN 2500
) -> List[Dict]:
    
    raw_text = extract_text_with_fitz(pdf_path)
    cleaned_text = clean_legal_pdf_text(raw_text)

    chapters = parse_chapters(cleaned_text)
    final_chunks = []

    for chapter in chapters:
        chapter_full = chapter["chapter_full"]
        articles = split_article_blocks(chapter["content"])

        for article in articles:
            article_title = article["article_title"]
            article_content = article["article_content"].strip()

            if not article_content:
                continue

            combined_title = f"{article_title} | {chapter_full}" if chapter_full else article_title

            base_payload = {
                "law_name": law_name,
                "source_file": source_file,
                "chapter": chapter_full,
                "article": article_title,
                "title": combined_title,
                "domain": domain,
                "effective_date": effective_date,
                "law_id": law_id
            }

            # === PHẦN SỬA CHÍNH: Giữ nguyên toàn bộ Điều ===
            sub_parts = split_long_article(article_content, max_context_len)

            for idx, part in enumerate(sub_parts, start=1):
                chunk = {
                    **base_payload,
                    "context": part.strip(),
                }
                if len(sub_parts) > 1:
                    chunk["sub_part"] = idx  # chỉ ghi sub_part khi thực sự tách
                final_chunks.append(chunk)

    return final_chunks


# =========================
# 6. SAVE OUTPUT
# =========================
def save_to_json(data: List[Dict], output_file: str):
    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)


# =========================
# 7. MAIN
# =========================
if __name__ == "__main__":
    pdf_path = r"E:\VN-legal-chatbot\data_processing\Luat_giao_thong.pdf"  # dùng raw string
    source_file = "Luat_giao_thong.pdf"
    law_name = "Luật Trật tự, an toàn giao thông đường bộ 2024"
    domain = "giao_thong"
    effective_date = "2025-01-01"
    law_id = "36/2024/QH15"
    output_json = "output_qdrant_ready_luat_giao_thong_new.json"

    chunks = process_legal_pdf(
        pdf_path=pdf_path,
        source_file=source_file,
        law_name=law_name,
        domain=domain,
        effective_date=effective_date,
        law_id=law_id,
        max_context_len=2500
    )

    print(f"Total chunks created: {len(chunks)}")
    if chunks:
        print("Sample chunk:")
        print(json.dumps(chunks[0], ensure_ascii=False, indent=2))

    save_to_json(chunks, output_json)
    print(f"Saved to {output_json}")

<>:95: SyntaxWarning: invalid escape sequence '\.'
<>:95: SyntaxWarning: invalid escape sequence '\.'
<>:95: SyntaxWarning: invalid escape sequence '\.'
<>:95: SyntaxWarning: invalid escape sequence '\.'
<>:95: SyntaxWarning: invalid escape sequence '\.'
<>:95: SyntaxWarning: invalid escape sequence '\.'
C:\Users\Admin\AppData\Local\Temp\ipykernel_24104\1740277802.py:95: SyntaxWarning: invalid escape sequence '\.'
  separators=["\n\n\n", "\n\n", "\n1\. ", "\n2\. ", "\n3\. ", "\n", ". ", " ", ""]
C:\Users\Admin\AppData\Local\Temp\ipykernel_24104\1740277802.py:95: SyntaxWarning: invalid escape sequence '\.'
  separators=["\n\n\n", "\n\n", "\n1\. ", "\n2\. ", "\n3\. ", "\n", ". ", " ", ""]
C:\Users\Admin\AppData\Local\Temp\ipykernel_24104\1740277802.py:95: SyntaxWarning: invalid escape sequence '\.'
  separators=["\n\n\n", "\n\n", "\n1\. ", "\n2\. ", "\n3\. ", "\n", ". ", " ", ""]
e:\VN-legal-chatbot\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update j

Total chunks created: 30
Sample chunk:
{
  "law_name": "Luật Trật tự, an toàn giao thông đường bộ 2024",
  "source_file": "Luat_giao_thong.pdf",
  "chapter": "Chương I NHỮNG QUY ĐỊNH CHUNG",
  "article": "Điều 1. Phạm vi điều chỉnh",
  "title": "Điều 1. Phạm vi điều chỉnh | Chương I NHỮNG QUY ĐỊNH CHUNG",
  "domain": "giao_thong",
  "effective_date": "2025-01-01",
  "law_id": "36/2024/QH15",
  "context": "Luật này quy định về quy tắc, phương tiện, người tham gia giao thông đường \nbộ, chỉ huy, điều khiển, tuần tra, kiểm soát, giải quyết tai nạn giao thông đường bộ, \ntrách nhiệm quản lý nhà nước và trách nhiệm của cơ quan, tổ chức, cá nhân có liên \nquan đến trật tự, an toàn giao thông đường bộ."
}
Saved to output_qdrant_ready_luat_giao_thong_new.json
